In [2]:
# -*- coding: utf-8 -*-
"""
ПОИСК ПАТТЕРНОВ С РЕГУЛЯРНЫМИ ВЫРАЖЕНИЯМИ
Мы НЕ знаем конкретные значения, но знаем ФОРМАТЫ
"""

import re

# ==================== ТЕКСТ ДЛЯ АНАЛИЗА (реальные данные) ====================

text = """
СЛУЖЕБНАЯ ЗАПИСКА

Сотрудник: Васильев Дмитрий Алексеевич
Паспорт: 5002 345678
СНИЛС: 987-654-321 00
ИНН: 123456789012
Телефон: +7-916-555-88-99
Email: dmitry.vasiliev@company.ru

Также упоминается: Елена Смирнова, паспорт 4611 223344, телефон 8 (495) 123-45-67

Документ имеет гриф: СЕКРЕТНО
"""

print("="*70)
print("ТЕКСТ ДЛЯ АНАЛИЗА (реальные данные, мы их НЕ знаем заранее):")
print("="*70)
print(text)

# ==================== ШАБЛОНЫ (то, что мы ЗНАЕМ заранее) ====================

# Мы не знаем конкретные номера, но знаем ИХ ФОРМАТЫ!
patterns = {
    "ФИО": {
        "regex": r"[А-Я][а-я]+ [А-Я][а-я]+(?: [А-Я][а-я]+)?",
        "penalty": 1,
        "description": "Имя Фамилия Отчество"
    },
    "Паспорт_РФ": {
        "regex": r"\d{4} \d{6}",
        "penalty": 8,
        "description": "Паспорт РФ (4 цифры пробел 6 цифр)"
    },
    "СНИЛС": {
        "regex": r"\d{3}-\d{3}-\d{3} \d{2}",
        "penalty": 10,
        "description": "СНИЛС (XXX-XXX-XXX XX)"
    },
    "ИНН_физ": {
        "regex": r"\d{12}",
        "penalty": 5,
        "description": "ИНН физического лица (12 цифр)"
    },
    "Телефон1": {
        "regex": r"\+7-\d{3}-\d{3}-\d{2}-\d{2}",
        "penalty": 3,
        "description": "Телефон +7-XXX-XXX-XX-XX"
    },
    "Телефон2": {
        "regex": r"8 \(\d{3}\) \d{3}-\d{2}-\d{2}",
        "penalty": 3,
        "description": "Телефон 8 (XXX) XXX-XX-XX"
    },
    "Email": {
        "regex": r"\S+@\S+\.\S+",
        "penalty": 2,
        "description": "Email адрес"
    },
    "Гриф_секретно": {
        "regex": r"СЕКРЕТНО|КОНФИДЕНЦИАЛЬНО|ДСП",
        "penalty": 20,
        "description": "Гриф секретности"
    }
}

# ==================== ПОИСК ПО ШАБЛОНАМ ====================

print("\n" + "="*70)
print("ПОИСК ПО ШАБЛОНАМ (регулярные выражения):")
print("="*70)

found_items = []
total_penalty = 0

for pattern_name, pattern_info in patterns.items():
    regex = pattern_info["regex"]
    penalty = pattern_info["penalty"]

    # Ищем ВСЕ совпадения в тексте
    matches = re.findall(regex, text)

    if matches:
        for match in matches:
            found_items.append({
                "type": pattern_name,
                "value": match,
                "penalty": penalty,
                "description": pattern_info["description"]
            })
            total_penalty += penalty
            print(f"\n✓ НАЙДЕНО: {pattern_name}")
            print(f"  Значение: '{match}'")
            print(f"  Штраф: {penalty}")
            print(f"  Описание: {pattern_info['description']}")

# ==================== КОНТЕКСТНЫЕ МОДИФИКАТОРЫ ====================

print("\n" + "="*70)
print("БАЗОВЫЙ ШТРАФ:")
print("="*70)
print(f"Сумма штрафов за все найденные данные: {total_penalty}")

context_multiplier = 1.0
reasons = []

# Проверяем наличие грифа секретности
if any(item["type"] == "Гриф_секретно" for item in found_items):
    context_multiplier *= 3
    reasons.append("найден гриф секретности (×3)")

# Проверяем комбинацию паспорт + СНИЛС
has_passport = any("Паспорт" in item["type"] for item in found_items)
has_snils = any("СНИЛС" in item["type"] for item in found_items)
if has_passport and has_snils:
    context_multiplier *= 2
    reasons.append("паспорт + СНИЛС вместе (×2)")

# Проверяем комбинацию ФИО + паспорт
has_name = any("ФИО" in item["type"] for item in found_items)
if has_name and has_passport:
    context_multiplier *= 1.5
    reasons.append("ФИО + паспорт вместе (×1.5)")

print("\n" + "="*70)
print("КОНТЕКСТНЫЕ МОДИФИКАТОРЫ:")
print("="*70)
if reasons:
    for reason in reasons:
        print(f"  • {reason}")
else:
    print("  • Нет дополнительных множителей")
print(f"\nИтоговый множитель: ×{context_multiplier}")

# ==================== ИТОГОВЫЙ РАСЧЁТ ====================

final_penalty = total_penalty * context_multiplier

print("\n" + "="*70)
print("ИТОГОВЫЙ ШТРАФ:")
print("="*70)
print(f"  Базовый штраф: {total_penalty}")
print(f"  × Множитель: {context_multiplier}")
print(f"  = {final_penalty:.1f} баллов")

# ==================== ВЕРДИКТ ====================

print("\n" + "="*70)
print("ВЕРДИКТ:")
print("="*70)

if final_penalty >= 50:
    print("🚨 КРИТИЧЕСКАЯ УГРОЗА! Документ ЗАБЛОКИРОВАН.")
    print("   → Обнаружены особо опасные данные")
    print("   → Уведомление службе безопасности")
elif final_penalty >= 20:
    print("⚠️ ВЫСОКИЙ РИСК! Требуется проверка.")
    print("   → Отправить на модерацию")
elif final_penalty >= 10:
    print("⚡ ПОВЫШЕННОЕ ВНИМАНИЕ! Документ помечен.")
    print("   → Логирование события")
else:
    print("✅ Безопасно. Пропустить.")

print("="*70)

# ==================== ВЫВОД ВСЕХ НАЙДЕННЫХ ДАННЫХ ====================

print("\n" + "="*70)
print("СВОДКА ПО НАЙДЕННЫМ ДАННЫМ:")
print("="*70)
for item in found_items:
    print(f"  [{item['penalty']} баллов] {item['type']}: {item['value']}")

ТЕКСТ ДЛЯ АНАЛИЗА (реальные данные, мы их НЕ знаем заранее):

СЛУЖЕБНАЯ ЗАПИСКА

Сотрудник: Васильев Дмитрий Алексеевич
Паспорт: 5002 345678
СНИЛС: 987-654-321 00
ИНН: 123456789012
Телефон: +7-916-555-88-99
Email: dmitry.vasiliev@company.ru

Также упоминается: Елена Смирнова, паспорт 4611 223344, телефон 8 (495) 123-45-67

Документ имеет гриф: СЕКРЕТНО


ПОИСК ПО ШАБЛОНАМ (регулярные выражения):

✓ НАЙДЕНО: ФИО
  Значение: 'Васильев Дмитрий Алексеевич'
  Штраф: 1
  Описание: Имя Фамилия Отчество

✓ НАЙДЕНО: ФИО
  Значение: 'Елена Смирнова'
  Штраф: 1
  Описание: Имя Фамилия Отчество

✓ НАЙДЕНО: Паспорт_РФ
  Значение: '5002 345678'
  Штраф: 8
  Описание: Паспорт РФ (4 цифры пробел 6 цифр)

✓ НАЙДЕНО: Паспорт_РФ
  Значение: '4611 223344'
  Штраф: 8
  Описание: Паспорт РФ (4 цифры пробел 6 цифр)

✓ НАЙДЕНО: СНИЛС
  Значение: '987-654-321 00'
  Штраф: 10
  Описание: СНИЛС (XXX-XXX-XXX XX)

✓ НАЙДЕНО: ИНН_физ
  Значение: '123456789012'
  Штраф: 5
  Описание: ИНН физического лица (12 цифр)

